# Fine-Tune Qwen 2.5 Coder (3B) on TPU (v5e-1)

This notebook covers fine-tuning on a **TPU**.  
**Note:** `bitsandbytes` (4-bit quantization) does NOT support TPUs. We will use native **BFloat16** training with **LoRA** and the **Adafactor** optimizer to save memory.

### ⚠️ Runtime Setting
Ensure your Runtime type is set to **TPU v5e-1**.

In [ ]:
# Install TPU-compatible libraries
# ⚠️ CRITICAL: Versions of torch and torch_xla MUST match exactly.
!pip install -q -U "torch~=2.5.1" "torch_xla[tpu]~=2.5.1" transformers peft trl datasets accelerate
# Note: No bitsandbytes needed

In [ ]:
# ------------------------------------------------------------------
# Verification Step: Check TPU Availability
# ------------------------------------------------------------------
try:
    import torch
    import torch_xla
    import torch_xla.core.xla_model as xm
    
    device = xm.xla_device()
    print("✅ TPU Device Found:", device)
    print("✅ Torch Version:", torch.__version__)
    print("✅ Torch XLA Version:", torch_xla.__version__)
except ImportError:
    print("❌ TPU Library NOT found. Please check dependencies.")
except Exception as e:
    print(f"❌ TPU Initialization Error: {e}")

## 2. Upload Dataset

Please upload the `sql_to_mql_finetuning.jsonl` file.

In [ ]:
from google.colab import files
import os

if not os.path.exists('sql_to_mql_finetuning.jsonl'):
    print("Please upload 'sql_to_mql_finetuning.jsonl'.")
    uploaded = files.upload()

## 3. Fine-Tuning Script (TPU Native)

Run Training on TPU.

In [ ]:
# Explicit imports to ensure registration
import torch
import torch_xla
import torch_xla.core.xla_model as xm

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import os

# Ensure XLA (TPU) is visible to libtpu
os.environ["PJRT_DEVICE"] = "TPU"

# Config
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
DATA_PATH = "sql_to_mql_finetuning.jsonl"
NEW_MODEL = "Qwen2.5-Coder-3B-Instruct-mql-adapter"

# 1. Load Model in BFloat16 (Native TPU type)
print("Loading model in BFloat16...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # <--- Native TPU format
    device_map=None,             # Let XLA handle devices via 'dilemma' (automap)
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable() 

# 2. LoRA Config
peft_config = LoraConfig(
    r=32,            
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Apply PEFT
base_model = get_peft_model(base_model, peft_config)
base_model.print_trainable_parameters()

# 3. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. Data
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_chat_template(row):
    conversation = row["messages"]
    text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat_template)
print(f"Dataset Size: {len(dataset)}")

# 5. SFT Config (TPU Optimized)
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,   # Try batch 4 on TPU
    gradient_accumulation_steps=2,
    optim="adafactor",               # Adafactor is safer for memory
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,                       # Enable BFloat16
    fp16=False,                      
    max_grad_norm=0.3,
    group_by_length=True,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

# 6. Trainer
# Note: No special 'device' arg needed; accelerate detects TPUs if torch_xla is present
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting Training on TPU...")
trainer.train()

print("Saving model...")
trainer.model.save_pretrained(NEW_MODEL)
tokenizer.save_pretrained(NEW_MODEL)
print("Training Complete")

## 4. Save to Drive

Mount Google Drive and save the adapter for persistent use.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest_path = f"/content/drive/MyDrive/{NEW_MODEL}"
shutil.copytree(NEW_MODEL, dest_path)
print(f"Model saved to {dest_path}")